# 3 · Gedächtnis und Rhythmus

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/DE/jupyter/markov-book-de/03_gedaechtnis_und_rhythmus.ipynb)

Zwei Fragen in diesem Kapitel:

1. Wie viel **Gedächtnis** soll die Kette haben?
2. Wie bringen wir ihr **Rhythmus** bei, nicht nur Tonhöhen?

In [ ]:
# Einrichtung -- läuft von selbst (beim ersten Mal etwa 20 Sekunden)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # welche Version von GreasyPidgin verwendet wird (Git-Branch oder -Tag)
TUTORIAL_REF  = "main"   # welche Version der Hilfsdatei dieses Tutorials verwendet wird
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/DE/jupyter/markov-book-de/workshop_helpers.py",
        "workshop_helpers.py")

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

wh.set_language("de")    # deutsche Meldungen, Beschriftungen und Notennamen
print("Bereit!")

In [ ]:
midi_path = wh.choose_midi()
melody = wh.load_melody(midi_path)
print(melody)

## Gedächtnis: Kopieren oder Erfinden

Für jede Ordnung lernen wir eine Kette, erzeugen eine Melodie und messen, wie viel davon eine **wörtliche Kopie**
des Originals ist. `copy_ratio` ist der Anteil der 4-Noten-Gruppen der neuen Melodie, die unverändert auch im
Original vorkommen. (`random.seed` macht das Experiment wiederholbar.)

In [ ]:
results = {}

for order in (1, 2, 3, 4):
    chain = MarkovChain()
    chain.generateRulesFromList(melody.pitches, order=order)

    random.seed(3)
    seq = chain.getNElements(seed=tuple(melody.pitches[:order]), n=48)
    results[order] = seq

    ratio = wh.copy_ratio(melody.pitches, seq, n=4)
    print(f"Ordnung {order}:  {len(chain.rules):3d} Regeln   kopiert: {ratio:4.0%}")

Mit mehr Gedächtnis hat die Kette mehr Regeln, aber jede Regel weniger Auswahl. Irgendwann spielt sie
nur noch das Original ab. **Niedrige Ordnung: überraschend, oft unzusammenhängend. Hohe Ordnung: zusammenhängend, aber eine Kopie.**

Hör dir die Extreme an:

In [ ]:
for order in (1, 4):
    print(f"Ordnung {order}")
    wh.render_midi(results[order], melody.deltas, melody.bpm, f"order{order}.mid")
    wh.listen(f"order{order}.mid")

### Die Regeln als Bild

Ordnung 1 und 2 nebeneinander. Bei Ordnung 2 ist jede Zeile ein *Notenpaar*.
Zeilen mit nur einer hellen Zelle sind Stellen, an denen die Kette keine Wahl hat.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, order in zip(axes, (1, 2)):
    chain = MarkovChain()
    chain.generateRulesFromList(melody.pitches, order=order)
    wh.plot_transition_matrix(chain, ax=ax)
    ax.set_title(f"Ordnung {order}")
plt.tight_layout()
plt.show()

## Rhythmus: eine zweite Kette

Rhythmus ist nur eine weitere Folge von Symbolen: die Zeit zwischen den Notenanfängen in Schlägen
(`melody.deltas`). Also lernen wir eine **zweite Kette** dafür, genau wie die erste.

In [ ]:
rhythm_chain = MarkovChain()
rhythm_chain.generateRulesFromList(melody.deltas, order=2)

pitch_chain = MarkovChain()
pitch_chain.generateRulesFromList(melody.pitches, order=2)

N = 32
random.seed(None)
new_pitches = pitch_chain.getNElements(seed=tuple(melody.pitches[:2]), n=N)
new_rhythm  = rhythm_chain.getNElements(seed=tuple(melody.deltas[:2]), n=N)

print("Tonhöhen:", " ".join(wh.pitch_name(p) for p in new_pitches[:12]), "...")
print("Rhythmus:", new_rhythm[:12], "...")

In [ ]:
wh.render_midi(new_pitches, new_rhythm, melody.bpm, "separate_chains.mid")
wh.plot_notes(new_pitches, new_rhythm, title="Zwei getrennte Ketten", color="C1")
plt.show()
wh.listen("separate_chains.mid")

Die beiden Ketten wissen nichts voneinander: Jede Tonhöhe kann auf jedem Rhythmus landen.

## Eine Kette für beides: Symbole können alles sein

Ein Symbol muss keine Zahl sein. Wenn wir jede Tonhöhe mit ihrem Rhythmus zu einem Paar `(Tonhöhe, Delta)` verkleben
und eine Kette über diese Paare lernen, merkt sich die Kette, **welcher Rhythmus zu welcher Tonhöhe gehörte**.
Beachte, dass der Startwert eine *Liste* von Paaren ist (zwei, weil die Ordnung 2 ist).

In [ ]:
pairs = list(zip(melody.pitches, melody.deltas))
print(pairs[:4])

joint_chain = MarkovChain()
joint_chain.generateRulesFromList(pairs, order=2)

joint = joint_chain.getNElements(seed=pairs[:2], n=N)
joint_pitches = [p for p, d in joint]
joint_rhythm  = [d for p, d in joint]

In [ ]:
wh.render_midi(joint_pitches, joint_rhythm, melody.bpm, "joint_chain.mid")
wh.plot_notes(joint_pitches, joint_rhythm, title="Eine Kette für Tonhöhe und Rhythmus", color="C2")
plt.show()
wh.listen("joint_chain.mid")

Vergleiche die drei Versionen nach Gehör: geborgter Rhythmus (Kapitel 2), getrennte Ketten, gemeinsame Kette.
Die gemeinsame Kette behält die charakteristischen Kombinationen aus Ton und Rhythmus des Originals; die getrennten
Ketten mischen sie freier durcheinander.

**Probier es aus:**

- Nimm für die gemeinsame Kette `order=1`. Was ändert sich?
- Lerne die gemeinsame Kette mit `order=3`. Unterscheidet sich das Ergebnis noch vom Original?
- Welche Kombination (Ordnungen für Tonhöhe und Rhythmus) gefällt dir am besten?